# Entendimento inicial do dataset de voos

Este notebook explora o CSV derivado do Voo Regular Ativo (VRA) da ANAC. Ele não treina modelos. O objetivo é verificar estrutura, cobertura, alvos, valores ausentes, distribuição de atrasos e outliers antes de definir as variáveis preditoras.

Para gerar a base local, execute `py scripts/preparar_dados_v2.py` na raiz do projeto. O CSV é local e não é versionado.

### Preparação do ambiente e localização da base

Esta célula importa as bibliotecas, configura a exibição das tabelas e localiza a pasta `data` a partir da raiz do projeto.

In [5]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 100)

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'data').exists():
    ROOT = ROOT.parent
DATA_PATH = ROOT / 'data' / 'voos_vra_derivados.csv'
if not DATA_PATH.exists():
    raise FileNotFoundError(f'Arquivo não encontrado: {DATA_PATH}. Execute o script de preparação primeiro.')
DATA_PATH

PosixPath('/home/falberto/Git/mcdia/mcdia-ml-previsao-atraso-voos/data/voos_vra_derivados.csv')

### Carregamento do dataset

Agora lemos o CSV derivado e convertemos os quatro campos de data e hora para o tipo de data do pandas.

In [2]:
date_columns = ['partida_prevista', 'partida_real', 'chegada_prevista', 'chegada_real']
df = pd.read_csv(DATA_PATH, parse_dates=date_columns, low_memory=False)
print('Dimensões:', df.shape)
df.head(3)

Dimensões: (1992832, 27)


,companhia_icao,empresa,numero_voo,codigo_di,codigo_tipo_linha,modelo_equipamento,numero_assentos,origem_icao,origem_descricao,partida_prevista,partida_real,destino_icao,destino_descricao,chegada_prevista,chegada_real,situacao_voo,situacao_partida,situacao_chegada,codeshare,cancelado,realizado,atraso_partida_min,atraso_chegada_min,atraso_partida_15m,atraso_chegada_15m,arquivo_origem,linha_origem
0,AAL,"AMERICAN AIRLINES, INC.",0904,0,I,B772,288,SBGL,AEROPORTO INTERNACIONAL DO RIO DE JANEIRO (GAL...,2024-01-01 23:55:00,2024-01-01 23:47:00,KMIA,"MIAMI INTERNATIONAL AIRPORT - MIAMI, FLORIDA -...",2024-01-02 07:45:00,2024-01-02 08:19:00,REALIZADO,Antecipado,Atraso 30-60,GLO/6004,False,True,-8.0,34.0,False,True,VRA_2024_01.csv,1
1,AAL,"AMERICAN AIRLINES, INC.",0905,0,I,B772,288,KMIA,"MIAMI INTERNATIONAL AIRPORT - MIAMI, FLORIDA -...",2024-01-01 23:55:00,2024-01-01 01:29:00,SBGL,AEROPORTO INTERNACIONAL DO RIO DE JANEIRO (GAL...,2024-01-02 09:25:00,2024-01-01 09:35:00,REALIZADO,Antecipado,Antecipado,GLO/6005,False,True,-1346.0,-1430.0,False,False,VRA_2024_01.csv,2
2,AAL,"AMERICAN AIRLINES, INC.",0906,0,I,B77W,318,SBGR,GUARULHOS - GOVERNADOR ANDRÉ FRANCO MONTORO - ...,2024-01-01 00:55:00,2024-01-01 00:46:00,KMIA,"MIAMI INTERNATIONAL AIRPORT - MIAMI, FLORIDA -...",2024-01-01 08:35:00,2024-01-01 08:45:00,REALIZADO,Antecipado,Pontual,GLO/6006,False,True,-9.0,10.0,False,False,VRA_2024_01.csv,3


## 1. Exploração rápida para iniciantes

Depois de carregar uma base, é útil fazer algumas perguntas simples: quantas linhas e colunas existem, como são os primeiros registros, quais tipos de dados foram reconhecidos e quais são as principais estatísticas numéricas. Estas operações são descritivas: ajudam a entender a base antes de formular modelos.

### Tamanho e primeiras linhas

shape informa o número de linhas e colunas. head() mostra os primeiros registros para conferirmos se os dados foram carregados como esperado.

In [3]:
print('Linhas e colunas:', df.shape)
df.head()

Linhas e colunas: (1992832, 27)


,companhia_icao,empresa,numero_voo,codigo_di,codigo_tipo_linha,modelo_equipamento,numero_assentos,origem_icao,origem_descricao,partida_prevista,partida_real,destino_icao,destino_descricao,chegada_prevista,chegada_real,situacao_voo,situacao_partida,situacao_chegada,codeshare,cancelado,realizado,atraso_partida_min,atraso_chegada_min,atraso_partida_15m,atraso_chegada_15m,arquivo_origem,linha_origem
0,AAL,"AMERICAN AIRLINES, INC.",0904,0,I,B772,288,SBGL,AEROPORTO INTERNACIONAL DO RIO DE JANEIRO (GAL...,2024-01-01 23:55:00,2024-01-01 23:47:00,KMIA,"MIAMI INTERNATIONAL AIRPORT - MIAMI, FLORIDA -...",2024-01-02 07:45:00,2024-01-02 08:19:00,REALIZADO,Antecipado,Atraso 30-60,GLO/6004,False,True,-8.0,34.0,False,True,VRA_2024_01.csv,1
1,AAL,"AMERICAN AIRLINES, INC.",0905,0,I,B772,288,KMIA,"MIAMI INTERNATIONAL AIRPORT - MIAMI, FLORIDA -...",2024-01-01 23:55:00,2024-01-01 01:29:00,SBGL,AEROPORTO INTERNACIONAL DO RIO DE JANEIRO (GAL...,2024-01-02 09:25:00,2024-01-01 09:35:00,REALIZADO,Antecipado,Antecipado,GLO/6005,False,True,-1346.0,-1430.0,False,False,VRA_2024_01.csv,2
2,AAL,"AMERICAN AIRLINES, INC.",0906,0,I,B77W,318,SBGR,GUARULHOS - GOVERNADOR ANDRÉ FRANCO MONTORO - ...,2024-01-01 00:55:00,2024-01-01 00:46:00,KMIA,"MIAMI INTERNATIONAL AIRPORT - MIAMI, FLORIDA -...",2024-01-01 08:35:00,2024-01-01 08:45:00,REALIZADO,Antecipado,Pontual,GLO/6006,False,True,-9.0,10.0,False,False,VRA_2024_01.csv,3
3,AAL,"AMERICAN AIRLINES, INC.",0925,0,I,B772,288,KMIA,"MIAMI INTERNATIONAL AIRPORT - MIAMI, FLORIDA -...",2024-01-01 21:20:00,2024-01-01 23:17:00,SBGR,GUARULHOS - GOVERNADOR ANDRÉ FRANCO MONTORO - ...,2024-01-02 07:50:00,2024-01-02 07:47:00,REALIZADO,Atraso 60-120,Antecipado,NaN,False,True,117.0,-3.0,True,False,VRA_2024_01.csv,4
4,AAL,"AMERICAN AIRLINES, INC.",0929,0,I,B77W,318,KMIA,"MIAMI INTERNATIONAL AIRPORT - MIAMI, FLORIDA -...",2024-01-01 20:50:00,2024-01-01 21:51:00,SBGR,GUARULHOS - GOVERNADOR ANDRÉ FRANCO MONTORO - ...,2024-01-02 06:20:00,2024-01-02 06:13:00,REALIZADO,Atraso 60-120,Antecipado,GLO/6007,False,True,61.0,-7.0,True,False,VRA_2024_01.csv,5


### Tipos das colunas e memória usada

info() resume o tipo de cada coluna, a quantidade de valores preenchidos e o uso aproximado de memória. É uma verificação importante para descobrir, por exemplo, se uma data foi lida como data ou como texto.

In [ ]:
df.info()

### Estatísticas numéricas com describe()

describe() calcula automaticamente estatísticas das colunas numéricas: quantidade (count), média (mean), desvio padrão (std), mínimo, quartis e máximo. O primeiro quartil é 25%, a mediana é 50% e o terceiro quartil é 75%.

In [ ]:
df.describe().T

### Média, mediana e outras medidas explicitamente

Também podemos pedir as medidas diretamente com agg(). A média pode ser influenciada por atrasos muito grandes; por isso comparamos com a mediana, que representa melhor o valor central quando há outliers.

In [ ]:
colunas_numericas = ['atraso_partida_min', 'atraso_chegada_min']
resumo_central = df[colunas_numericas].agg(['count', 'mean', 'median', 'std', 'min', 'max']).T
resumo_central

### Contagem de categorias

Para colunas categóricas, value_counts() mostra quantos registros pertencem a cada categoria. Aqui verificamos a situação operacional dos voos.

In [ ]:
df['situacao_voo'].value_counts(dropna=False)

## 2. Schema e qualidade básica

As colunas de horários reais e situações operacionais são rótulos ou informações posteriores. Elas não devem entrar como preditoras no primeiro experimento.

### Dicionário de tipos e valores ausentes

Esta tabela mostra o tipo de cada coluna e quantos valores estão ausentes. Use-a para identificar quais campos podem ser usados com segurança.

In [ ]:
schema = pd.DataFrame({'tipo': df.dtypes.astype(str), 'ausentes': df.isna().sum(), 'percentual_ausente': (100 * df.isna().mean()).round(2)})
schema

### Duplicidades e situações operacionais

Aqui verificamos registros repetidos e observamos quantos voos foram realizados ou cancelados, além das categorias de situação da chegada.

In [ ]:
print('Duplicidades:', int(df.duplicated().sum()))
print('Arquivos de origem:', df['arquivo_origem'].value_counts().sort_index().to_dict())
print('Situação do voo:')
display(df['situacao_voo'].value_counts(dropna=False).to_frame('quantidade'))
print('Situação da chegada:')
display(df['situacao_chegada'].value_counts(dropna=False).to_frame('quantidade'))

## 3. Alvos iniciais

`cancelado` é um alvo separado. Para atraso de chegada, usamos somente voos realizados com horário previsto e real válidos. O primeiro alvo recomendado é `atraso_chegada_15m`, uma classificação binária.

### Resumo dos alvos

Esta célula quantifica cancelamentos, voos realizados e o número de casos em que o atraso de chegada pode ser observado.

In [ ]:
target_summary = pd.DataFrame({
    'quantidade': [df['cancelado'].sum(), df['realizado'].sum(), df['atraso_chegada_15m'].notna().sum(), df['atraso_chegada_15m'].sum()],
}, index=['cancelados', 'realizados', 'alvos_de_atraso_observáveis', 'chegadas_15m_ou_mais_atrasadas'])
target_summary

### Estatísticas dos atrasos

Calculamos percentis do atraso de partida e de chegada apenas para voos realizados. Os valores negativos representam antecipação.

In [ ]:
realizados = df[df['realizado']].copy()
realizados[['atraso_partida_min', 'atraso_chegada_min']].describe(percentiles=[.01, .05, .25, .5, .75, .95, .99]).round(2)

## 4. Distribuição e outliers

A amostra já revelou registros com atrasos superiores a 24 horas e casos acima de 30 dias. Esses valores não devem ser removidos automaticamente. O alvo binário é menos sensível a esse problema; uma regressão em minutos exigirá uma regra de tratamento definida antes da avaliação.

### Investigação de outliers

Esta tabela lista os maiores atrasos de chegada. Observe especialmente os casos acima de 24 horas antes de decidir qualquer tratamento.

In [ ]:
outliers = realizados[realizados['atraso_chegada_min'] > 24 * 60].sort_values('atraso_chegada_min', ascending=False)
print('Atrasos de chegada acima de 24 horas:', len(outliers))
outliers[['arquivo_origem', 'companhia_icao', 'numero_voo', 'origem_icao', 'destino_icao', 'chegada_prevista', 'chegada_real', 'atraso_chegada_min', 'situacao_chegada']].head(20)

### Visualização da distribuição

Os gráficos mostram a distribuição dos atrasos em uma faixa limitada para facilitar a leitura e o equilíbrio do alvo binário de 15 minutos.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
realizados['atraso_chegada_min'].clip(-60, 360).plot.hist(bins=60, ax=axes[0], title='Atraso de chegada (limitado a -60 a 360 min)')
realizados['atraso_chegada_15m'].value_counts().sort_index().plot.bar(ax=axes[1], title='Alvo: chegada com atraso >= 15 min')
axes[0].set_xlabel('Minutos')
axes[1].set_xlabel('False / True')
plt.tight_layout()

## 5. Padrões por tempo, companhia e aeroporto

Estas tabelas são descritivas. Elas não demonstram causalidade e não devem ser usadas diretamente como estimativas do desempenho futuro sem uma divisão temporal.

### Evolução por mês

Criamos variáveis simples de calendário e resumimos volume, taxa de atraso e mediana por mês de referência.

In [ ]:
realizados['mes_previsto'] = realizados['partida_prevista'].dt.to_period('M').astype(str)
realizados['dia_semana'] = realizados['partida_prevista'].dt.day_name()
realizados['hora_prevista'] = realizados['partida_prevista'].dt.hour
monthly = realizados.groupby('mes_previsto').agg(voos=('realizado', 'size'), taxa_atraso_15m=('atraso_chegada_15m', 'mean'), mediana_atraso_min=('atraso_chegada_min', 'median')).reset_index()
monthly

### Comparação por companhia

Esta tabela compara companhias com pelo menos 100 voos. Ela é descritiva e não deve ser interpretada como ranking de desempenho.

In [ ]:
by_carrier = realizados.groupby('companhia_icao').agg(voos=('realizado', 'size'), taxa_atraso_15m=('atraso_chegada_15m', 'mean'), mediana_atraso_min=('atraso_chegada_min', 'median')).query('voos >= 100').sort_values('taxa_atraso_15m', ascending=False)
by_carrier.head(20)

### Comparação por aeroporto e hora

Por fim, observamos taxas por aeroporto de origem e por hora prevista de partida, sempre como exploração inicial e sem inferir causalidade.

In [ ]:
by_origin = realizados.groupby('origem_icao').agg(voos=('realizado', 'size'), taxa_atraso_15m=('atraso_chegada_15m', 'mean')).query('voos >= 100').sort_values('taxa_atraso_15m', ascending=False)
by_hour = realizados.groupby('hora_prevista').agg(voos=('realizado', 'size'), taxa_atraso_15m=('atraso_chegada_15m', 'mean')).reset_index()
display(by_origin.head(20))
display(by_hour)

## Conclusão provisória

O dataset é grande em número de voos, mas ainda cobre apenas dois meses. O próximo notebook deve construir variáveis preditoras disponíveis antes do voo, definir a janela temporal de treino/validação/teste e comparar baselines simples. As taxas acima são exploratórias e não representam previsão operacional.